Project Name - Gender Classification Model

Project Type - Productionization of ML Systems

Contribution - Individual

Name of Contributor - Meetu Kumari

GitHub Link -  https://github.com/MeetuKumari1/ml_productionization.git

Brief overview of dataset

Hotels Dataset:

* travelCode: Identifier for the travel, similar to the Flights dataset.
* userCode: User identifier(linked to the Users dataset)
* name: Name of the hotel.
* place: Location of the hotel.
* days: Number of days of the hotel stay.
* price: Price per day.
* total: Total price for the stay.
* date: Date of the hotel booking.

Project Objectives

Build a recommendation model to provide hotel suggestions based on user preferences and historical data.

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Dataset loading
from pathlib import Path
import zipfile

zip_path = Path("C:/Users/Coditas-Admin/Documents/ml_productionization/travel_capstone.zip")
extract_dir = zip_path.parent

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_dir)

print(f"Extracted to: {extract_dir}")

# for google drive (colab)
# zip_path = Path("/content/drive/MyDrive/Colab Notebooks/AB/spec/M1/project/travel_capstone.zip")
# extract_dir = zip_path.parent
# with zipfile.ZipFile(zip_path, "r") as zf:
#     zf.extractall(extract_dir)


Extracted to: C:\Users\Coditas-Admin\Documents\ml_productionization


In [3]:
# load flight dataset
hotels_df = pd.read_csv(
    "C:/Users/Coditas-Admin/Documents/ml_productionization/travel_capstone/hotels.csv"
)

In [4]:
# Dataset Rows & Columns count
print(f"Total number of rows: {hotels_df.shape[0]}")
print(f"Total number of columns: {hotels_df.shape[1]}")

Total number of rows: 40552
Total number of columns: 8


In [5]:
hotels_df.head()

,travelCode,userCode,name,place,days,price,total,date
0,0,0,Hotel A,Florianopolis (SC),4,313.02,1252.08,09/26/2019
1,2,0,Hotel K,Salvador (BH),2,263.41,526.82,10/10/2019
2,7,0,Hotel K,Salvador (BH),3,263.41,790.23,11/14/2019
3,11,0,Hotel K,Salvador (BH),4,263.41,1053.64,12/12/2019
4,13,0,Hotel A,Florianopolis (SC),1,313.02,313.02,12/26/2019


In [6]:
hotels_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40552 entries, 0 to 40551
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   travelCode  40552 non-null  int64  
 1   userCode    40552 non-null  int64  
 2   name        40552 non-null  object 
 3   place       40552 non-null  object 
 4   days        40552 non-null  int64  
 5   price       40552 non-null  float64
 6   total       40552 non-null  float64
 7   date        40552 non-null  object 
dtypes: float64(2), int64(3), object(3)
memory usage: 2.5+ MB


In [7]:
hotels_df.duplicated().value_counts()

False    40552
Name: count, dtype: int64

In [8]:
hotels_df.isnull().sum()

travelCode    0
userCode      0
name          0
place         0
days          0
price         0
total         0
date          0
dtype: int64

In [9]:
hotels_df.describe()

,travelCode,userCode,days,price,total
count,40552.000000,40552.000000,40552.000000,40552.000000,40552.000000
mean,67911.794461,666.963726,2.499679,214.439554,536.229513
std,39408.199333,391.136794,1.119326,76.742305,319.331482
min,0.000000,0.000000,1.000000,60.390000,60.390000
25%,33696.750000,323.000000,1.000000,165.990000,247.620000
50%,67831.000000,658.000000,2.000000,242.880000,495.240000
75%,102211.250000,1013.000000,4.000000,263.410000,742.860000
max,135942.000000,1339.000000,4.000000,313.020000,1252.080000


In [11]:
hotels_df.describe(include=['object'])

,name,place,date
count,40552,40552,40552
unique,9,9,199
top,Hotel K,Salvador (BH),10/03/2019
freq,5094,5094,404


In [13]:
def build_recommender(df: pd.DataFrame) -> dict:
    if df[["userCode", "name"]].isna().any().any():
        raise ValueError("Missing values in userCode/name.")

    interactions = pd.crosstab(df["userCode"], df["name"])
    user_ids = interactions.index.to_list()
    hotel_names = interactions.columns.to_list()

    item_matrix = interactions.T.values
    item_similarity = cosine_similarity(item_matrix)

    hotel_stats = (
        df.groupby(["name", "place"], as_index=False)
        .agg(bookings=("travelCode", "count"), avg_price=("price", "mean"))
        .sort_values(["bookings", "avg_price"], ascending=[False, True])
    )

    return {
        "user_ids": user_ids,
        "hotel_names": hotel_names,
        "item_similarity": item_similarity,
        "interactions": interactions,
        "hotel_stats": hotel_stats,
    }

In [14]:
HOTELS_PATH = Path("travel_capstone/hotels.csv")
USERS_PATH = Path("travel_capstone/users.csv")
ARTIFACT_PATH = Path("artifacts/hotel_recommender.joblib")
METADATA_PATH = Path("artifacts/hotel_recommender_metadata.json")

In [15]:
def save_recommender(recommender: dict) -> dict:
    ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(recommender, ARTIFACT_PATH)

    metadata = {
        "artifact_path": str(ARTIFACT_PATH),
        "num_users": len(recommender["user_ids"]),
        "num_hotels": len(recommender["hotel_names"]),
    }
    METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return metadata

In [16]:
def save_recommender(recommender: dict) -> dict:
    ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(recommender, ARTIFACT_PATH)

    metadata = {
        "artifact_path": str(ARTIFACT_PATH),
        "num_users": len(recommender["user_ids"]),
        "num_hotels": len(recommender["hotel_names"]),
    }
    METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return metadata

In [17]:
def train_recommender(data_path: str | Path = HOTELS_PATH) -> dict:
    df = hotels_df(data_path)
    recommender = build_recommender(df)
    return save_recommender(recommender)


def load_recommender() -> dict:
    if not ARTIFACT_PATH.exists():
        raise FileNotFoundError(
            f"Recommender artifact not found at {ARTIFACT_PATH}. "
            "Run train_recommender() first."
        )
    return joblib.load(ARTIFACT_PATH)


In [18]:
def recommend_for_user(recommender: dict,user_code: int,top_n: int = 5,) -> pd.DataFrame:
    interactions = recommender["interactions"]
    hotel_stats = recommender["hotel_stats"]
    hotel_names = recommender["hotel_names"]
    item_similarity = recommender["item_similarity"]

    if user_code not in interactions.index:
        return hotel_stats.head(top_n)

    user_vector = interactions.loc[user_code].values
    scores = user_vector @ item_similarity
    seen_mask = user_vector > 0
    scores = scores.astype(float)
    scores[seen_mask] = -np.inf

    top_idx = np.argsort(scores)[::-1][:top_n]
    recommended_hotels = [hotel_names[i] for i in top_idx]

    return hotel_stats[hotel_stats["name"].isin(recommended_hotels)]

In [22]:
train_recommender(HOTELS_PATH)

TypeError: 'DataFrame' object is not callable